# Aho–Corasick 多模式串匹配

把非空模式插入字典树，再按 BFS 构建失配指针。失配指针指向当前串最长且仍出现在树中的真后缀。扫描文本时，输出当前位置命中的模式及其后缀模式。这里预先合并节点的输出列表以简化查找，深度嵌套的模式会占用额外存储。

In [ ]:
from collections import deque


def multi_find(text, patterns):
    children = [{}]
    outputs = [[]]
    failure = [0]
    for pattern in dict.fromkeys(patterns):
        if not pattern:
            raise ValueError("模式串不能为空")
        node = 0
        for char in pattern:
            if char not in children[node]:
                children[node][char] = len(children)
                children.append({})
                outputs.append([])
                failure.append(0)
            node = children[node][char]
        outputs[node].append(pattern)

    queue = deque(children[0].values())
    while queue:
        node = queue.popleft()
        for char, child in children[node].items():
            fallback = failure[node]
            while fallback and char not in children[fallback]:
                fallback = failure[fallback]
            failure[child] = children[fallback].get(char, 0)
            outputs[child].extend(outputs[failure[child]])
            queue.append(child)

    matches = []
    node = 0
    for end, char in enumerate(text):
        while node and char not in children[node]:
            node = failure[node]
        node = children[node].get(char, 0)
        matches.extend((end - len(pattern) + 1, pattern) for pattern in outputs[node])
    return matches


assert multi_find("ushers", ["he", "she", "his", "hers"]) == [(1, "she"), (2, "he"), (2, "hers")]
assert multi_find("aaa", ["a", "aa"]) == [(0, "a"), (0, "aa"), (1, "a"), (1, "aa"), (2, "a")]
assert multi_find("text", []) == []